<a href="https://colab.research.google.com/github/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%204/4.3%20Using%20Tools%20and%20Memory/1%20Tutorial%20-%20Using%20Tools%20(Search%2C%20Calculator)%20Openrouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install
!pip install -q langchain-core==1.6.3 langchain-openai==1.6.2 ddgs==9.16.0 numexpr==2.14.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 104.6 MB/s eta 0:00:00


## Tutorial: Using LangChain Tools (Search, Calculator)
We’ll register two tools and call them via a tiny router (no heavy agents yet).


In [2]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

# Enter your OpenRouter API key securely when prompted.
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")

# OpenRouter provides an OpenAI-compatible API.
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# You can change this to any compatible OpenRouter model.
MODEL = "openai/gpt-4o-mini"

llm = ChatOpenAI(
    model=MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0,
    seed=42,
)

print("OpenRouter configured successfully.")
print("Model:", MODEL)
from langchain_core.tools import Tool

# Reuse the OpenRouter-configured model above.


Enter your OpenRouter API key: ··········
OpenRouter configured successfully.
Model: openai/gpt-4o-mini


### Step 1: Define tools
We’ll add a calculator and a web search (DuckDuckGo) with narrow scopes.


In [3]:
import numexpr as ne
from ddgs import DDGS


def calc(expression: str) -> str:
    try:
        value = ne.evaluate(expression)
        return str(value.item())
    except Exception as e:
        return f"Error: {e}"


def web_search(query: str, max_results: int = 3) -> str:
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=max_results))
    # Return a compact string for the LLM; in production, pass structured results
    lines = [f"- {r.get('title')}: {r.get('href')}" for r in results]
    return "\n".join(lines)[:1200]


calc_tool = Tool(name="calculator", func=calc, description="Evaluate simple math expressions, return a number as string.")
search_tool = Tool(name="search", func=web_search, description="DuckDuckGo web search, returns top links.")

print(calc_tool.run("(3+4)*2"))
print(search_tool.run("LangChain framework overview"))


14
- LangChain - Wikipedia: https://en.wikipedia.org/wiki/LangChain
- What is LangChain? - LangChain Explained - AWS: https://aws.amazon.com/what-is/langchain/
- What Is LangChain? | IBM: https://www.ibm.com/think/topics/langchain


### Step 2: A tiny “tool-aware” responder
We’ll route based on input shape. This mirrors agent behavior without the overhead.


In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

explain_prompt = PromptTemplate.from_template(
    "Explain briefly: {question}"
)
explain_chain = explain_prompt | llm | StrOutputParser()


def route(query: str) -> str:
    looks_math = any(ch.isdigit() for ch in query) and any(op in query for op in "+-*/()")
    looks_search = any(word in query.lower() for word in ["latest", "current", "news", "what is", "overview"]) and not looks_math
    if looks_math:
        return calc_tool.run(query)
    if looks_search:
        return search_tool.run(query)
    return explain_chain.invoke({"question": query})

print(route("(12+8)/5"))
print(route("latest on langchain memory"))
print(route("What is a PromptTemplate?"))


4.0
- Memory - Docs by LangChain: https://docs.langchain.com/oss/python/deepagents/memory
- Memory overview - Docs by LangChain: https://docs.langchain.com/oss/python/concepts/memory
- How LangChain Enhances AI Memory for Better User Experiences ...: https://www.geeky-gadgets.com/conversational-memory-langchain/
- Agentforce Notes. What are prompt templates ? (Hint … | by Ajinkya Phadnis | Medium: https://medium.com/@ajinkyamails/agentforce-specialist-certification-9a849060f77a
- What is a Prompt Template? | AI21: https://www.ai21.com/glossary/foundational-llm/prompt-template/
- What is a Prompt template? | PromptLayer: https://www.promptlayer.com/glossary/prompt-template/
